# Notebook 23: Calibrated Graph-Bayes Rescue Reranker

Offline lab for a larger algorithmic-ledger improvement over Notebook 13.

Notebook 23 keeps Notebook 13 as the first-pass live workup, then adds a calibrated mathematical final layer:

1. train/validate-derived synthetic partial evidence states for candidate scoring;
2. an L2-regularized candidate reranker;
3. deterministic graph/Bayes rescue certificates;
4. at most three extra graph-Bayes discriminator requests only for suspicious final states.

No API calls are made. The selected policy is fixed before evaluating the 49-case saved trace, and the 49-case labels are used only for final evaluation and paired error analysis.

## 1. Utility Functions

In [ ]:
from __future__ import annotations

import ast
import json
import math
import random
import time
import warnings
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if not (ROOT / "notebooks").exists():
    ROOT = ROOT.parent

SEED = 2919
rng = np.random.default_rng(SEED)
random.seed(SEED)

ARTIFACT_ROOT = ROOT / "artifacts" / "graph_algorithmic_ledger" / "calibrated_graph_bayes_rescue_reranker_49case_v1"
FIGURE_DIR = ARTIFACT_ROOT / "figures"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK13_ROOT = ROOT / "artifacts" / "sequential_hybrid_mlp_feedback" / "selected_stop_live_confirmation_49case_v1"
NOTEBOOK22_ROOT = ROOT / "artifacts" / "graph_algorithmic_ledger" / "graph_posterior_final_adjudicator_49case_v1"
GRAPH_ROOT = ROOT / "artifacts" / "graph_algorithmic_ledger" / "medkgi_style_offline_notebook13_49case_v1"
BAYES_ROOT = ROOT / "artifacts" / "bayesian_voi_ledger" / "bayesian_voi_offline_notebook13_49case_v1"
PARTIAL_MLP_ROOT = ROOT / "artifacts" / "one_shot_partial_evidence" / "partial_evidence_one_shot_final_policy_masked_v2"
DATASET_ROOT = ROOT / "dataset"

SYNTH_TRAIN_NROWS = 80_000
SYNTH_VALIDATE_NROWS = 40_000
SYNTH_TRAIN_STATES = 8_000
SYNTH_VALIDATE_STATES = 4_000
MAX_RESCUE_REQUESTS = 3
TOTAL_REQUEST_CAP = 24
RERANKER_TIE_GRAPH_EPS = 0.02

SELECTED_POLICY_NAME = "calibrated_graph_bayes_rescue_v1"
FEATURE_CSV_NAME = "candidate_level_train_validate_features.csv"

def load_json(path: Path) -> Any:
    return json.loads(path.read_text())


def safe_parse_list(value: Any) -> list[Any]:
    if isinstance(value, list):
        return value
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except Exception:
        pass
    try:
        parsed = ast.literal_eval(str(value))
        return list(parsed) if isinstance(parsed, (list, tuple)) else []
    except Exception:
        try:
            parsed = json.loads(str(value))
            return list(parsed) if isinstance(parsed, list) else []
        except Exception:
            return []


def parse_evidence_token(token: str) -> tuple[str, str | None]:
    token = str(token)
    if "_@_" in token:
        root_id, value = token.split("_@_", 1)
        return root_id, value
    return token, None


def row_evidence_by_root(row: dict[str, Any]) -> dict[str, list[str]]:
    evidence_by_root: dict[str, list[str]] = {}
    for token in safe_parse_list(row.get("EVIDENCES", [])):
        root_id, value = parse_evidence_token(str(token))
        evidence_by_root.setdefault(root_id, []).append("present" if value is None else str(value))
    return evidence_by_root


def initial_root(row_or_value: Any) -> str:
    value = row_or_value.get("INITIAL_EVIDENCE") if isinstance(row_or_value, dict) else row_or_value
    tokens = safe_parse_list(value)
    token = tokens[0] if tokens else str(value)
    root_id, _ = parse_evidence_token(token)
    return root_id


def load_split(split: str, nrows: int | None = None) -> pd.DataFrame:
    path = DATASET_ROOT / f"release_{split}_patients.zip"
    with zipfile.ZipFile(path) as zf:
        member = zf.namelist()[0]
        with zf.open(member) as f:
            return pd.read_csv(f, nrows=nrows)


def stable_softmax(scores: np.ndarray) -> np.ndarray:
    arr = np.asarray(scores, dtype=float)
    exps = np.exp(arr - np.max(arr))
    denom = float(exps.sum())
    return exps / denom if denom else np.ones_like(arr) / len(arr)


def normalized_entropy(probs: np.ndarray) -> float:
    arr = np.asarray(probs, dtype=float)
    arr = arr[arr > 0]
    if len(arr) == 0:
        return 0.0
    return float(-(arr * np.log(arr)).sum() / math.log(max(2, len(probs))))


def rank_order(scores: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    order = np.argsort(-np.asarray(scores))
    ranks = np.empty_like(order)
    ranks[order] = np.arange(1, len(order) + 1)
    return order, ranks


def unique_extend(items: list[str]) -> list[str]:
    out: list[str] = []
    for item in items:
        if item and item not in out:
            out.append(item)
    return out

print("Project root:", ROOT)
print("Artifact root:", ARTIFACT_ROOT)

## 2. Load Inputs And Models

In [ ]:
evidences = load_json(DATASET_ROOT / "release_evidences.json")
conditions = load_json(DATASET_ROOT / "release_conditions.json")

notebook13_predictions = pd.read_csv(NOTEBOOK13_ROOT / "predictions.csv")
notebook22_case_results = pd.read_csv(NOTEBOOK22_ROOT / "case_level_graph_adjudicator_results.csv")
notebook22_case_results = notebook22_case_results[
    (notebook22_case_results["run_scope"] == "notebook13_49case")
    & (notebook22_case_results["policy_name"] == "conservative_graph_critic_v1_clip3_margin1")
].copy()
notebook22_graph_features = pd.read_csv(NOTEBOOK22_ROOT / "graph_final_state_features.csv")
notebook22_graph_features = notebook22_graph_features[notebook22_graph_features["run_scope"] == "notebook13_49case"].copy()

with (NOTEBOOK13_ROOT / "traces.jsonl").open() as f:
    notebook13_traces = {json.loads(line)["case_id"]: json.loads(line) for line in f}

request_distribution = notebook13_predictions["num_requests"].astype(int).to_numpy()
label_names = list(torch.load(PARTIAL_MLP_ROOT / "best_model.pt", map_location="cpu")["label_names"])
label_to_index = {label: idx for idx, label in enumerate(label_names)}
all_roots = list(evidences.keys())

print("Notebook 13 predictions:", notebook13_predictions.shape)
print("Notebook 22 selected case results:", notebook22_case_results.shape)
print("Notebook 22 graph features:", notebook22_graph_features.shape)
print("Trace count:", len(notebook13_traces))
print("Labels:", len(label_names), "Roots:", len(all_roots))

In [ ]:
def encode_age(age: int) -> int:
    age = int(age)
    if age < 1:
        return 0
    if age <= 4:
        return 1
    if age <= 14:
        return 2
    if age <= 29:
        return 3
    if age <= 44:
        return 4
    if age <= 59:
        return 5
    if age <= 74:
        return 6
    return 7


def encode_sex(sex: str) -> int:
    return 0 if str(sex) == "M" else 1


@dataclass
class ObservationSchema:
    root_ids: list[str]
    slot_slices: dict[str, tuple[int, int]]
    data_types: dict[str, str]
    possible_values: dict[str, list[str]]
    default_values: dict[str, str | None]
    categorical_integer_roots: set[str]
    feature_names: list[str]

    @classmethod
    def from_metadata(cls, metadata: dict[str, dict[str, Any]]) -> "ObservationSchema":
        root_ids = list(metadata.keys())
        slot_slices: dict[str, tuple[int, int]] = {}
        data_types: dict[str, str] = {}
        possible_values: dict[str, list[str]] = {}
        default_values: dict[str, str | None] = {}
        categorical_integer_roots: set[str] = set()
        feature_names = [f"age_bin_{idx}" for idx in range(8)] + ["sex_M", "sex_F"]
        cursor = 10
        for root_id, meta in metadata.items():
            data_type = meta.get("data_type", "B")
            raw_values = meta.get("possible-values", [])
            values = [str(value) for value in raw_values]
            default_value = meta.get("default_value")
            default_value = None if default_value is None else str(default_value)
            data_types[root_id] = data_type
            possible_values[root_id] = values
            default_values[root_id] = default_value
            if data_type == "B":
                slot_slices[root_id] = (cursor, cursor + 1)
                feature_names.append(root_id)
                cursor += 1
            elif data_type == "C":
                if raw_values and not isinstance(raw_values[0], str):
                    categorical_integer_roots.add(root_id)
                    slot_slices[root_id] = (cursor, cursor + 1)
                    feature_names.append(root_id)
                    cursor += 1
                else:
                    slot_slices[root_id] = (cursor, cursor + len(values))
                    feature_names.extend(f"{root_id}__{value}" for value in values)
                    cursor += len(values)
            elif data_type == "M":
                slot_slices[root_id] = (cursor, cursor + len(values))
                feature_names.extend(f"{root_id}__{value}" for value in values)
                cursor += len(values)
            else:
                raise ValueError(f"Unsupported evidence type {data_type} for {root_id}")
        return cls(root_ids, slot_slices, data_types, possible_values, default_values, categorical_integer_roots, feature_names)

    @property
    def feature_size(self) -> int:
        return len(self.feature_names)

    def initial_state(self, age: int, sex: str) -> np.ndarray:
        state = np.zeros(self.feature_size, dtype=np.float32)
        state[encode_age(age)] = 1.0
        state[8 + encode_sex(sex)] = 1.0
        return state

    def apply_root_observation(self, state: np.ndarray, root_id: str, present_values: list[str] | None = None) -> np.ndarray:
        if root_id not in self.slot_slices:
            return state
        values = [str(value) for value in (present_values or [])]
        data_type = self.data_types[root_id]
        start, end = self.slot_slices[root_id]
        default_value = self.default_values[root_id]
        if data_type == "B":
            state[start] = 1.0 if values else -1.0
            return state
        if root_id in self.categorical_integer_roots:
            chosen = values[0] if values else default_value
            if chosen is None:
                state[start] = -1.0
            else:
                possible = self.possible_values[root_id]
                denominator = max(1, len(possible) - 1)
                state[start] = float(possible.index(str(chosen))) / denominator if str(chosen) in possible else -1.0
            return state
        state[start:end] = -1.0
        for value in values:
            if value in self.possible_values[root_id]:
                state[start + self.possible_values[root_id].index(value)] = 1.0
        return state


class DirectDiagnosisMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes: list[int], num_classes: int, dropout: float = 0.0):
        super().__init__()
        layers: list[nn.Module] = []
        previous_dim = input_dim
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(previous_dim, hidden_size))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            previous_dim = hidden_size
        self.backbone = nn.Sequential(*layers)
        self.classifier = nn.Linear(previous_dim, num_classes)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.backbone(features))


schema = ObservationSchema.from_metadata(evidences)
checkpoint = torch.load(PARTIAL_MLP_ROOT / "best_model.pt", map_location="cpu")
resolved_mlp_config = checkpoint.get("resolved_run_config", {})
partial_mlp = DirectDiagnosisMLP(
    schema.feature_size,
    list(resolved_mlp_config.get("hidden_sizes", [2048, 2048, 2048])),
    len(label_names),
    float(resolved_mlp_config.get("dropout", 0.0)),
)
partial_mlp.load_state_dict(checkpoint["model_state_dict"])
partial_mlp.eval()


def encode_row_state(row: dict[str, Any], revealed_roots: set[str]) -> np.ndarray:
    evidence_by_root = row_evidence_by_root(row)
    state = schema.initial_state(int(row["AGE"]), str(row["SEX"]))
    for root_id in sorted(revealed_roots):
        schema.apply_root_observation(state, root_id, evidence_by_root.get(root_id, []))
    return state


def mlp_probs_for_states(rows: list[dict[str, Any]], revealed_root_sets: list[set[str]], batch_size: int = 512) -> np.ndarray:
    features = np.stack([encode_row_state(row, roots) for row, roots in zip(rows, revealed_root_sets)])
    batches: list[np.ndarray] = []
    with torch.no_grad():
        for start in range(0, len(features), batch_size):
            logits = partial_mlp(torch.from_numpy(features[start : start + batch_size]))
            batches.append(torch.softmax(logits, dim=1).numpy())
    return np.vstack(batches)

print("Partial MLP feature size:", schema.feature_size)

## 3. Graph-Bayes State Scoring

In [ ]:
# Status-level graph weights. This intentionally matches Notebook 22's robust present/absent final-state scoring.
graph_edges = pd.read_csv(GRAPH_ROOT / "global_evidence_graph_edges.csv")
graph_edges = graph_edges[graph_edges["outcome_state"].isin(["present", "absent"])].copy()
graph_weight = {
    (str(row.root_evidence_id), str(row.outcome_state), str(row.pathology)): float(np.clip(row.log_odds_support, -3, 3))
    for row in graph_edges.itertuples()
}

# Status-level Bayesian likelihoods from Notebook 19. These are used for posterior and rescue-question utility.
bayes_likelihoods = pd.read_csv(BAYES_ROOT / "root_outcome_likelihoods.csv")
bayes_likelihoods = bayes_likelihoods[bayes_likelihoods["outcome_state"].isin(["__ABSENT__", "__PRESENT__"])].copy()
bayes_prob: dict[tuple[str, str, str], float] = {}
for _, row in bayes_likelihoods.iterrows():
    state = "absent" if row["outcome_state"] == "__ABSENT__" else "present"
    for label in label_names:
        bayes_prob[(str(row["root_evidence_id"]), state, label)] = float(row[f"p__{label}"])


def state_outcomes(row: dict[str, Any], roots: set[str]) -> list[tuple[str, str]]:
    evidence_by_root = row_evidence_by_root(row)
    return [(root_id, "present" if evidence_by_root.get(root_id) else "absent") for root_id in sorted(roots)]


def graph_bayes_scores(row: dict[str, Any], roots: set[str]) -> dict[str, np.ndarray]:
    graph_scores = np.zeros(len(label_names), dtype=float)
    bayes_log_scores = np.zeros(len(label_names), dtype=float)
    for root_id, state in state_outcomes(row, roots):
        for idx, label in enumerate(label_names):
            graph_scores[idx] += graph_weight.get((root_id, state, label), 0.0)
            bayes_log_scores[idx] += math.log(max(1e-6, bayes_prob.get((root_id, state, label), 0.5)))
    graph_posterior = stable_softmax(graph_scores)
    bayes_posterior = stable_softmax(bayes_log_scores)
    graph_order, graph_rank = rank_order(graph_scores)
    bayes_order, bayes_rank = rank_order(bayes_posterior)
    return {
        "graph_scores": graph_scores,
        "graph_posterior": graph_posterior,
        "graph_order": graph_order,
        "graph_rank": graph_rank,
        "bayes_log_scores": bayes_log_scores,
        "bayes_posterior": bayes_posterior,
        "bayes_order": bayes_order,
        "bayes_rank": bayes_rank,
    }


def ranked_from_probs(probs: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    return rank_order(probs)


def candidate_feature_rows(
    case_id: str,
    split: str,
    true_pathology: str,
    reference_pred: str,
    prior_pred: str,
    row: dict[str, Any],
    roots: set[str],
    mlp_probs: np.ndarray,
    llm_ranked: list[str] | None = None,
    source: str = "synthetic",
) -> list[dict[str, Any]]:
    llm_ranked = llm_ranked or []
    scores = graph_bayes_scores(row, roots)
    graph_scores = scores["graph_scores"]
    graph_posterior = scores["graph_posterior"]
    graph_order = scores["graph_order"]
    graph_rank = scores["graph_rank"]
    bayes_log_scores = scores["bayes_log_scores"]
    bayes_posterior = scores["bayes_posterior"]
    bayes_order = scores["bayes_order"]
    bayes_rank = scores["bayes_rank"]
    mlp_order, mlp_rank = ranked_from_probs(mlp_probs)
    mlp_top_conf = float(mlp_probs[mlp_order[0]])
    mlp_second = float(mlp_probs[mlp_order[1]])
    mlp_entropy = normalized_entropy(mlp_probs)
    candidate_pool = unique_extend(
        [reference_pred, prior_pred]
        + [label_names[idx] for idx in graph_order[:5]]
        + [label_names[idx] for idx in bayes_order[:5]]
        + [label_names[idx] for idx in mlp_order[:5]]
        + list(llm_ranked[:5])
    )
    rows = []
    for candidate in candidate_pool:
        idx = label_to_index[candidate]
        llm_rr = 0.0
        if candidate in llm_ranked[:5]:
            llm_rr = 1.0 / (1 + llm_ranked[:5].index(candidate))
        rows.append(
            {
                "case_id": case_id,
                "split": split,
                "source": source,
                "true_pathology": true_pathology,
                "candidate": candidate,
                "reference_pred": reference_pred,
                "prior_pred": prior_pred,
                "label": int(candidate == true_pathology),
                "is_reference": int(candidate == reference_pred),
                "is_prior": int(candidate == prior_pred),
                "graph_rank": int(graph_rank[idx]),
                "graph_score": float(graph_scores[idx]),
                "graph_posterior": float(graph_posterior[idx]),
                "graph_margin_to_top": float(graph_scores[graph_order[0]] - graph_scores[idx]),
                "graph_is_top1": int(graph_rank[idx] == 1),
                "bayes_rank": int(bayes_rank[idx]),
                "bayes_log_score": float(bayes_log_scores[idx]),
                "bayes_posterior": float(bayes_posterior[idx]),
                "bayes_margin_to_top": float(bayes_posterior[bayes_order[0]] - bayes_posterior[idx]),
                "mlp_rank": int(mlp_rank[idx]),
                "mlp_posterior": float(mlp_probs[idx]),
                "mlp_margin_to_top": float(mlp_probs[mlp_order[0]] - mlp_probs[idx]),
                "mlp_top_conf": mlp_top_conf,
                "mlp_margin": float(mlp_top_conf - mlp_second),
                "mlp_entropy": float(mlp_entropy),
                "llm_reciprocal_rank": float(llm_rr),
                "num_visible_roots": int(len(roots)),
            }
        )
    return rows

print("Graph status edges:", len(graph_edges))
print("Bayes status likelihood rows:", len(bayes_likelihoods))

## 4. Synthetic Train/Validate Candidate States

In [ ]:
def sample_revealed_roots(row: dict[str, Any], requested_count: int) -> set[str]:
    evidence_by_root = row_evidence_by_root(row)
    init = initial_root(row)
    requested_count = int(max(0, min(TOTAL_REQUEST_CAP, requested_count)))
    sampled: set[str] = set()
    present_roots = [root for root in evidence_by_root.keys() if root != init]
    if present_roots and requested_count > 0:
        take_present = min(len(present_roots), max(0, requested_count // 2))
        if take_present:
            sampled.update(rng.choice(present_roots, size=take_present, replace=False).tolist())
    remaining = requested_count - len(sampled)
    if remaining > 0:
        candidates = [root for root in all_roots if root != init and root not in sampled]
        sampled.update(rng.choice(candidates, size=min(len(candidates), remaining), replace=False).tolist())
    return {init} | sampled


def make_synthetic_states(frame: pd.DataFrame, n_states: int, split: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    if len(frame) > n_states:
        work = frame.sample(n=n_states, random_state=SEED).reset_index(drop=True)
    else:
        work = frame.reset_index(drop=True)
    rows = [row.to_dict() for _, row in work.iterrows()]
    request_counts = [int(rng.choice(request_distribution)) for _ in rows]
    revealed_sets = [sample_revealed_roots(row, count) for row, count in zip(rows, request_counts)]
    initial_sets = [{initial_root(row)} for row in rows]
    mlp_final_probs = mlp_probs_for_states(rows, revealed_sets)
    mlp_initial_probs = mlp_probs_for_states(rows, initial_sets)
    feature_rows: list[dict[str, Any]] = []
    state_rows: list[dict[str, Any]] = []
    for idx, (row, roots, init_probs, final_probs, req_count) in enumerate(zip(rows, revealed_sets, mlp_initial_probs, mlp_final_probs, request_counts)):
        reference_pred = label_names[int(np.argmax(final_probs))]
        prior_pred = label_names[int(np.argmax(init_probs))]
        case_id = f"{split}:{idx}"
        feature_rows.extend(
            candidate_feature_rows(
                case_id=case_id,
                split=split,
                true_pathology=str(row["PATHOLOGY"]),
                reference_pred=reference_pred,
                prior_pred=prior_pred,
                row=row,
                roots=roots,
                mlp_probs=final_probs,
                llm_ranked=[],
                source="synthetic_partial_state",
            )
        )
        state_rows.append(
            {
                "case_id": case_id,
                "split": split,
                "true_pathology": row["PATHOLOGY"],
                "reference_pred": reference_pred,
                "prior_pred": prior_pred,
                "requested_count": int(req_count),
                "visible_root_count": int(len(roots)),
                "reference_correct": int(reference_pred == row["PATHOLOGY"]),
                "prior_correct": int(prior_pred == row["PATHOLOGY"]),
            }
        )
    return pd.DataFrame(feature_rows), pd.DataFrame(state_rows)

start = time.time()
train_frame = load_split("train", nrows=SYNTH_TRAIN_NROWS)
validate_frame = load_split("validate", nrows=SYNTH_VALIDATE_NROWS)
train_features, train_state_summary = make_synthetic_states(train_frame, SYNTH_TRAIN_STATES, "train_synthetic")
validate_features, validate_state_summary = make_synthetic_states(validate_frame, SYNTH_VALIDATE_STATES, "validate_synthetic")
synthetic_features = pd.concat([train_features, validate_features], ignore_index=True)
synthetic_state_summary = pd.concat([train_state_summary, validate_state_summary], ignore_index=True)

synthetic_summary = pd.DataFrame(
    [
        {
            "split": "train_synthetic",
            "states": len(train_state_summary),
            "candidate_rows": len(train_features),
            "reference_accuracy": train_state_summary["reference_correct"].mean(),
            "prior_accuracy": train_state_summary["prior_correct"].mean(),
            "mean_visible_roots": train_state_summary["visible_root_count"].mean(),
        },
        {
            "split": "validate_synthetic",
            "states": len(validate_state_summary),
            "candidate_rows": len(validate_features),
            "reference_accuracy": validate_state_summary["reference_correct"].mean(),
            "prior_accuracy": validate_state_summary["prior_correct"].mean(),
            "mean_visible_roots": validate_state_summary["visible_root_count"].mean(),
        },
    ]
)

synthetic_features.to_csv(ARTIFACT_ROOT / FEATURE_CSV_NAME, index=False)
synthetic_summary.to_csv(ARTIFACT_ROOT / "synthetic_state_generation_summary.csv", index=False)
print("Synthetic features:", synthetic_features.shape)
print("Synthetic generation seconds:", round(time.time() - start, 2))
display(synthetic_summary)

## 5. L2 Candidate Reranker And Validation Calibration

In [ ]:
FEATURE_COLUMNS = [
    "is_reference",
    "is_prior",
    "graph_rank",
    "graph_score",
    "graph_posterior",
    "graph_margin_to_top",
    "graph_is_top1",
    "bayes_rank",
    "bayes_log_score",
    "bayes_posterior",
    "bayes_margin_to_top",
    "mlp_rank",
    "mlp_posterior",
    "mlp_margin_to_top",
    "mlp_top_conf",
    "mlp_margin",
    "mlp_entropy",
    "llm_reciprocal_rank",
    "num_visible_roots",
]

train_X = train_features[FEATURE_COLUMNS]
train_y = train_features["label"].astype(int)
validate_X = validate_features[FEATURE_COLUMNS]
validate_y = validate_features["label"].astype(int)

validation_rows = []
best_model = None
best_key = None
for C in [0.05, 0.10, 0.20, 0.50, 1.00]:
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=C, max_iter=1000, class_weight="balanced", random_state=SEED),
    )
    model.fit(train_X, train_y)
    scored = validate_features.copy()
    scored["reranker_score"] = model.predict_proba(validate_X)[:, 1]
    top_idx = scored.groupby("case_id")["reranker_score"].idxmax()
    selected = scored.loc[top_idx].copy()
    reference = scored[scored["is_reference"] == 1].drop_duplicates("case_id")
    top1_accuracy = float(selected["label"].mean())
    reference_accuracy = float(reference["label"].mean())
    validation_rows.append(
        {
            "model_name": f"l2_logistic_C{C}",
            "C": C,
            "reference_accuracy": reference_accuracy,
            "reranker_top1_accuracy": top1_accuracy,
            "delta_vs_reference": top1_accuracy - reference_accuracy,
        }
    )
    key = (top1_accuracy, -C)
    if best_key is None or key > best_key:
        best_key = key
        best_model = model
        best_C = C

assert best_model is not None
validation_summary = pd.DataFrame(validation_rows)

validate_scored = validate_features.copy()
validate_scored["reranker_score"] = best_model.predict_proba(validate_X)[:, 1]
validate_selected = validate_scored.loc[validate_scored.groupby("case_id")["reranker_score"].idxmax()].copy()
validate_reference_scores = validate_scored[validate_scored["is_reference"] == 1][["case_id", "candidate", "reranker_score", "label"]].rename(
    columns={"candidate": "reference_candidate", "reranker_score": "reference_score", "label": "reference_label"}
)
validate_selected = validate_selected.merge(validate_reference_scores, on="case_id", how="left")

threshold_rows = []
best_threshold_key = None
best_threshold = None
for min_score in np.linspace(0.05, 0.80, 16):
    for min_margin in np.linspace(-0.20, 0.50, 15):
        correct = []
        changes = improvements = regressions = 0
        for row in validate_selected.itertuples():
            override = (
                row.candidate != row.reference_candidate
                and row.reranker_score >= min_score
                and (row.reranker_score - row.reference_score) >= min_margin
            )
            row_correct = bool(row.label) if override else bool(row.reference_label)
            correct.append(row_correct)
            changes += int(override)
            improvements += int(override and bool(row.label) and not bool(row.reference_label))
            regressions += int(override and (not bool(row.label)) and bool(row.reference_label))
        accuracy = float(np.mean(correct))
        threshold_rows.append(
            {
                "min_score": float(min_score),
                "min_score_margin": float(min_margin),
                "accuracy": accuracy,
                "changes": int(changes),
                "improvements": int(improvements),
                "regressions": int(regressions),
            }
        )
        key = (accuracy, -regressions, -changes)
        if best_threshold_key is None or key > best_threshold_key:
            best_threshold_key = key
            best_threshold = (float(min_score), float(min_margin), accuracy, int(changes), int(improvements), int(regressions))

threshold_summary = pd.DataFrame(threshold_rows)
selected_threshold_summary = pd.DataFrame(
    [
        {
            "selected_C": best_C,
            "selected_min_score": best_threshold[0],
            "selected_min_score_margin": best_threshold[1],
            "selected_validate_accuracy": best_threshold[2],
            "selected_validate_changes": best_threshold[3],
            "selected_validate_improvements": best_threshold[4],
            "selected_validate_regressions": best_threshold[5],
        }
    ]
)
reranker_validation_summary = pd.concat([validation_summary, selected_threshold_summary], axis=0, ignore_index=True, sort=False)
reranker_validation_summary.to_csv(ARTIFACT_ROOT / "reranker_validation_summary.csv", index=False)
print("Selected logistic C:", best_C)
display(reranker_validation_summary)

## 6. Reconstruct Notebook 13 Final States

In [ ]:
def reconstruct_notebook13_roots(case_id: str, row: dict[str, Any]) -> set[str]:
    roots = {initial_root(row)}
    trace_payload = notebook13_traces[case_id]
    for turn in trace_payload.get("trace", []):
        reveal_payload = turn.get("reveal_payload")
        if reveal_payload:
            roots.add(str(reveal_payload["root_evidence_id"]))
    return roots


def test_rows_for_notebook13() -> list[dict[str, Any]]:
    test_frame = load_split("test")
    rows = []
    for pred_row in notebook13_predictions.itertuples():
        patient_row = test_frame.iloc[int(pred_row.source_row_index)].to_dict()
        roots = reconstruct_notebook13_roots(str(pred_row.case_id), patient_row)
        rows.append({"prediction": pred_row._asdict(), "patient": patient_row, "roots": roots})
    return rows

test_case_rows = test_rows_for_notebook13()
test_patient_rows = [item["patient"] for item in test_case_rows]
test_root_sets = [item["roots"] for item in test_case_rows]
test_mlp_probs = mlp_probs_for_states(test_patient_rows, test_root_sets, batch_size=256)

test_candidate_rows: list[dict[str, Any]] = []
for item, mlp_probs in zip(test_case_rows, test_mlp_probs):
    pred = item["prediction"]
    test_candidate_rows.extend(
        candidate_feature_rows(
            case_id=str(pred["case_id"]),
            split="notebook13_49case",
            true_pathology=str(pred["true_pathology"]),
            reference_pred=str(pred["predicted_pathology"]),
            prior_pred=str(pred.get("prior_top1", "")),
            row=item["patient"],
            roots=item["roots"],
            mlp_probs=mlp_probs,
            llm_ranked=safe_parse_list(pred.get("llm_ranked_differential", [])),
            source="notebook13_final_state",
        )
    )

test_candidates = pd.DataFrame(test_candidate_rows)
test_candidates["reranker_score"] = best_model.predict_proba(test_candidates[FEATURE_COLUMNS])[:, 1]
print("49-case candidate rows:", test_candidates.shape)

## 7. Rescue Continuation And Selected Policy

In [ ]:
def entropy_raw(probs: np.ndarray) -> float:
    arr = np.asarray(probs, dtype=float)
    arr = arr[arr > 0]
    return float(-(arr * np.log(arr)).sum()) if len(arr) else 0.0


def choose_rescue_root(row: dict[str, Any], roots: set[str], candidate_labels: list[str], q: np.ndarray) -> dict[str, Any] | None:
    base_entropy = entropy_raw(q)
    best: dict[str, Any] | None = None
    for root_id in all_roots:
        if root_id in roots:
            continue
        p_present = np.array([bayes_prob.get((root_id, "present", label), 0.5) for label in candidate_labels], dtype=float)
        p_absent = np.array([bayes_prob.get((root_id, "absent", label), 0.5) for label in candidate_labels], dtype=float)
        mass_present = float((q * p_present).sum())
        mass_absent = float((q * p_absent).sum())
        if mass_present + mass_absent <= 0:
            continue
        prob_present = mass_present / (mass_present + mass_absent)
        prob_absent = 1.0 - prob_present
        posterior_present = q * p_present
        posterior_absent = q * p_absent
        posterior_present = posterior_present / posterior_present.sum() if posterior_present.sum() > 0 else q
        posterior_absent = posterior_absent / posterior_absent.sum() if posterior_absent.sum() > 0 else q
        expected_entropy = prob_present * entropy_raw(posterior_present) + prob_absent * entropy_raw(posterior_absent)
        entropy_gain = base_entropy - expected_entropy
        pairwise_bonus = 0.0
        for i in range(min(3, len(candidate_labels))):
            for j in range(i + 1, min(5, len(candidate_labels))):
                pairwise_bonus = max(pairwise_bonus, abs(float(p_present[i] - p_present[j])))
        utility = entropy_gain + 0.05 * pairwise_bonus - 0.01
        if best is None or utility > best["utility"]:
            best = {
                "root_evidence_id": root_id,
                "question_en": evidences[root_id].get("question_en", root_id),
                "utility": float(utility),
                "entropy_gain": float(entropy_gain),
                "pairwise_bonus": float(pairwise_bonus),
            }
    return best


def post_rescue_reranker_values(
    case_id: str,
    true_pathology: str,
    row: dict[str, Any],
    roots: set[str],
    reference_pred: str,
    prior_pred: str,
    llm_ranked: list[str],
    rescue_probs: np.ndarray,
) -> list[dict[str, Any]]:
    mlp_order, _ = ranked_from_probs(rescue_probs)
    rescue_mlp_ranked = [label_names[idx] for idx in mlp_order[:5]]
    feature_rows = candidate_feature_rows(
        case_id=case_id,
        split="notebook13_49case_post_rescue",
        true_pathology=true_pathology,
        reference_pred=reference_pred,
        prior_pred=prior_pred,
        row=row,
        roots=roots,
        mlp_probs=rescue_probs,
        llm_ranked=llm_ranked,
        source="post_rescue_reranker",
    )
    frame = pd.DataFrame(feature_rows)
    if frame.empty:
        return []
    frame["reranker_score"] = best_model.predict_proba(frame[FEATURE_COLUMNS])[:, 1]
    frame["rescue_score"] = frame["reranker_score"]
    frame["candidate_source"] = "post_rescue_reranker"
    top_score = float(frame["reranker_score"].max())
    close = frame[frame["reranker_score"] >= top_score - RERANKER_TIE_GRAPH_EPS].copy()
    close = close.sort_values(["graph_rank", "graph_score", "reranker_score"], ascending=[True, False, False])
    rest = frame.drop(close.index).sort_values(["reranker_score", "graph_score"], ascending=[False, False])
    ranked = pd.concat([close, rest], ignore_index=True, sort=False)
    return ranked.to_dict("records")


def prior_recovery_certificate(features: pd.DataFrame, reference_pred: str, prior_pred: str) -> bool:
    if not prior_pred or prior_pred == reference_pred:
        return False
    ref = features[features["disease"] == reference_pred]
    prior = features[features["disease"] == prior_pred]
    if ref.empty or prior.empty:
        return False
    ref_row = ref.iloc[0]
    prior_row = prior.iloc[0]
    return bool(
        int(prior_row["graph_rank"]) <= 5
        and float(ref_row["graph_net_support"]) < 0.0
        and float(prior_row["graph_net_support"]) > float(ref_row["graph_net_support"])
    )


def graph_critic_certificate(features: pd.DataFrame, reference_pred: str) -> tuple[bool, str | None]:
    ordered = features.sort_values("graph_rank")
    top = ordered.iloc[0]
    second = ordered.iloc[1]
    ref = features[features["disease"] == reference_pred]
    if ref.empty:
        return False, None
    ref_score = float(ref.iloc[0]["graph_net_support"])
    margin = float(top["graph_net_support"] - second["graph_net_support"])
    ok = bool(
        str(top["disease"]) != reference_pred
        and margin >= 1.0
        and ref_score < 0.0
        and float(top["graph_net_support"]) > 0.0
    )
    return ok, str(top["disease"]) if ok else None


def rescue_trigger(row: dict[str, Any], roots: set[str]) -> bool:
    scores = graph_bayes_scores(row, roots)
    graph_top_posterior = float(scores["graph_posterior"][scores["graph_order"][0]])
    return bool(row["stop_reason"] == "agent_stop" and int(row["num_requests"]) <= 3 and graph_top_posterior < 0.80)


def apply_selected_policy() -> tuple[pd.DataFrame, pd.DataFrame, list[dict[str, Any]]]:
    case_rows = []
    candidate_rows = []
    trace_rows = []
    test_frame = load_split("test")
    for pred in notebook13_predictions.to_dict("records"):
        case_id = str(pred["case_id"])
        patient_row = test_frame.iloc[int(pred["source_row_index"])].to_dict()
        roots = reconstruct_notebook13_roots(case_id, patient_row)
        reference_pred = str(pred["predicted_pathology"])
        final_pred = reference_pred
        decision_source = "notebook13_reference"
        extra_roots: list[str] = []
        graph_features = notebook22_graph_features[notebook22_graph_features["case_id"] == case_id].copy()
        llm_ranked = safe_parse_list(pred.get("llm_ranked_differential", []))
        mlp_ranked = safe_parse_list(pred.get("mlp_ranked_differential", []))

        if prior_recovery_certificate(graph_features, reference_pred, str(pred.get("prior_top1", ""))):
            final_pred = str(pred["prior_top1"])
            decision_source = "prior_recovery_certificate"
        else:
            graph_ok, graph_pred = graph_critic_certificate(graph_features, reference_pred)
            if graph_ok and graph_pred is not None:
                final_pred = graph_pred
                decision_source = "conservative_graph_critic"
            elif rescue_trigger(pred, roots):
                decision_source = "rescue_abstained"
                rescue_roots = set(roots)
                for step in range(min(MAX_RESCUE_REQUESTS, TOTAL_REQUEST_CAP - int(pred["num_requests"]))):
                    scores = graph_bayes_scores(patient_row, rescue_roots)
                    graph_order = scores["graph_order"]
                    candidate_labels = unique_extend(
                        [reference_pred]
                        + [label_names[idx] for idx in graph_order[:5]]
                        + llm_ranked[:5]
                        + mlp_ranked[:5]
                    )
                    q = np.array([max(1e-9, scores["graph_posterior"][label_to_index[label]]) for label in candidate_labels], dtype=float)
                    q = q / q.sum()
                    chosen = choose_rescue_root(patient_row, rescue_roots, candidate_labels, q)
                    if chosen is None or chosen["utility"] <= 0:
                        break
                    root_id = chosen["root_evidence_id"]
                    rescue_roots.add(root_id)
                    extra_roots.append(root_id)
                    trace_rows.append(
                        {
                            "case_id": case_id,
                            "step": step + 1,
                            "root_evidence_id": root_id,
                            "question_en": chosen["question_en"],
                            "utility": chosen["utility"],
                            "entropy_gain": chosen["entropy_gain"],
                            "pairwise_bonus": chosen["pairwise_bonus"],
                            "revealed_status": "present" if row_evidence_by_root(patient_row).get(root_id) else "absent",
                        }
                    )
                if extra_roots:
                    rescue_probs = mlp_probs_for_states([patient_row], [rescue_roots], batch_size=1)[0]
                    values = post_rescue_reranker_values(
                        case_id=case_id,
                        true_pathology=str(pred["true_pathology"]),
                        row=patient_row,
                        roots=rescue_roots,
                        reference_pred=reference_pred,
                        prior_pred=str(pred.get("prior_top1", "")),
                        llm_ranked=llm_ranked,
                        rescue_probs=rescue_probs,
                    )
                    top = values[0] if values else None
                    ref_value = next((item for item in values if item["candidate"] == reference_pred), None)
                    accept = bool(
                        top is not None
                        and ref_value is not None
                        and top["candidate"] != reference_pred
                        and float(top["reranker_score"]) >= best_threshold[0]
                        and float(top["reranker_score"] - ref_value["reranker_score"]) >= best_threshold[1]
                        and int(top["graph_rank"]) <= 3
                        and (float(top["graph_score"] - ref_value["graph_score"]) > 0.75 or float(ref_value["graph_score"]) < 0.0)
                    )
                    if accept:
                        final_pred = str(top["candidate"])
                        decision_source = "graph_bayes_rescue_rerank"
                    for value in values:
                        candidate_rows.append(value)

        case_rows.append(
            {
                "case_id": case_id,
                "true_pathology": pred["true_pathology"],
                "notebook13_predicted_pathology": reference_pred,
                "notebook22_predicted_pathology": notebook22_case_results.set_index("case_id").loc[case_id, "graph_adjudicator_predicted_pathology"],
                "rescue_predicted_pathology": final_pred,
                "notebook13_correct": bool(pred["correct"]),
                "notebook22_correct": bool(notebook22_case_results.set_index("case_id").loc[case_id, "graph_adjudicator_correct"]),
                "rescue_correct": bool(final_pred == pred["true_pathology"]),
                "decision_source": decision_source,
                "num_requests_notebook13": int(pred["num_requests"]),
                "extra_requests": int(len(extra_roots)),
                "num_requests_rescue": int(pred["num_requests"] + len(extra_roots)),
                "extra_requested_roots": json.dumps(extra_roots),
                "stop_reason": pred["stop_reason"],
            }
        )
    return pd.DataFrame(case_rows), pd.DataFrame(candidate_rows), trace_rows

rescue_results, post_rescue_candidate_scores, rescue_trace_rows = apply_selected_policy()
win_loss = []
for row in rescue_results.itertuples():
    if row.notebook13_correct and row.rescue_correct:
        win_loss.append("both_correct")
    elif (not row.notebook13_correct) and row.rescue_correct:
        win_loss.append("rescue_only_correct")
    elif row.notebook13_correct and (not row.rescue_correct):
        win_loss.append("notebook13_only_correct")
    else:
        win_loss.append("both_wrong")
rescue_results["win_loss_vs_notebook13"] = win_loss
rescue_results["improvement_vs_notebook13"] = (~rescue_results["notebook13_correct"]) & rescue_results["rescue_correct"]
rescue_results["regression_vs_notebook13"] = rescue_results["notebook13_correct"] & (~rescue_results["rescue_correct"])

candidate_49 = pd.concat([test_candidates.copy(), post_rescue_candidate_scores], ignore_index=True, sort=False) if len(post_rescue_candidate_scores) else test_candidates.copy()
candidate_49.to_csv(ARTIFACT_ROOT / "candidate_level_49case_scores.csv", index=False)
rescue_results.to_csv(ARTIFACT_ROOT / "notebook13_rescue_case_results.csv", index=False)
rescue_results.to_csv(ARTIFACT_ROOT / "paired_notebook13_vs_rescue_reranker.csv", index=False)
with (ARTIFACT_ROOT / "rescue_trace.jsonl").open("w") as f:
    for row in rescue_trace_rows:
        f.write(json.dumps(row) + "\n")

print("Selected policy accuracy:", int(rescue_results["rescue_correct"].sum()), "/", len(rescue_results))
print("Extra requests total:", int(rescue_results["extra_requests"].sum()))
display(rescue_results[rescue_results["win_loss_vs_notebook13"] != "both_correct"])

## 8. Evaluation, Audits, And Promotion Decision

In [ ]:
def topk_from_serialized(value: Any, k: int) -> list[str]:
    return [str(item) for item in safe_parse_list(value)[:k]]

notebook13_correct = int(notebook13_predictions["correct"].sum())
notebook13_accuracy = float(notebook13_correct / len(notebook13_predictions))
notebook13_top3 = float(np.mean([row.true_pathology in topk_from_serialized(row.ranked_differential, 3) for row in notebook13_predictions.itertuples()]))
notebook13_top5 = float(np.mean([row.true_pathology in topk_from_serialized(row.ranked_differential, 5) for row in notebook13_predictions.itertuples()]))
notebook13_mean_requests = float(notebook13_predictions["num_requests"].mean())

notebook22_correct = int(notebook22_case_results["graph_adjudicator_correct"].sum())
notebook22_accuracy = float(notebook22_correct / len(notebook22_case_results))
notebook22_regressions = int(notebook22_case_results["regression_vs_notebook13"].sum())

rescue_correct = int(rescue_results["rescue_correct"].sum())
rescue_accuracy = float(rescue_correct / len(rescue_results))
rescue_mean_requests = float(rescue_results["num_requests_rescue"].mean())
rescue_extra_requests = int(rescue_results["extra_requests"].sum())
rescue_improvements = int(rescue_results["improvement_vs_notebook13"].sum())
rescue_regressions = int(rescue_results["regression_vs_notebook13"].sum())

policy_summary = pd.DataFrame(
    [
        {
            "system": "Notebook 13 selected-stop hybrid",
            "correct_count": notebook13_correct,
            "num_cases": len(notebook13_predictions),
            "accuracy": notebook13_accuracy,
            "top3_accuracy": notebook13_top3,
            "top5_accuracy": notebook13_top5,
            "mean_requests": notebook13_mean_requests,
            "extra_requests_total": 0,
            "improvements_vs_notebook13": 0,
            "regressions_vs_notebook13": 0,
        },
        {
            "system": "Notebook 22 conservative graph critic",
            "correct_count": notebook22_correct,
            "num_cases": len(notebook22_case_results),
            "accuracy": notebook22_accuracy,
            "top3_accuracy": np.nan,
            "top5_accuracy": np.nan,
            "mean_requests": notebook13_mean_requests,
            "extra_requests_total": 0,
            "improvements_vs_notebook13": int((~notebook22_case_results["notebook13_correct"] & notebook22_case_results["graph_adjudicator_correct"]).sum()),
            "regressions_vs_notebook13": notebook22_regressions,
        },
        {
            "system": "Notebook 23 calibrated graph-bayes rescue",
            "correct_count": rescue_correct,
            "num_cases": len(rescue_results),
            "accuracy": rescue_accuracy,
            "top3_accuracy": np.nan,
            "top5_accuracy": np.nan,
            "mean_requests": rescue_mean_requests,
            "extra_requests_total": rescue_extra_requests,
            "improvements_vs_notebook13": rescue_improvements,
            "regressions_vs_notebook13": rescue_regressions,
        },
    ]
)
policy_summary.to_csv(ARTIFACT_ROOT / "rescue_policy_summary.csv", index=False)

hard_cases = {}
for row in rescue_results[~rescue_results["notebook13_correct"] | (rescue_results["extra_requests"] > 0)].itertuples():
    graph_case = notebook22_graph_features[notebook22_graph_features["case_id"] == row.case_id].sort_values("graph_rank")
    hard_cases[row.case_id] = {
        "true_pathology": row.true_pathology,
        "notebook13_prediction": row.notebook13_predicted_pathology,
        "notebook22_prediction": row.notebook22_predicted_pathology,
        "notebook23_prediction": row.rescue_predicted_pathology,
        "decision_source": row.decision_source,
        "extra_requests": row.extra_requests,
        "extra_requested_roots": safe_parse_list(row.extra_requested_roots),
        "notebook13_correct": bool(row.notebook13_correct),
        "notebook23_correct": bool(row.rescue_correct),
        "top_graph_diagnoses_before_rescue": graph_case.head(10)[
            ["disease", "graph_rank", "graph_net_support", "graph_posterior", "is_true_pathology", "is_notebook13_prediction"]
        ].to_dict("records"),
    }
with (ARTIFACT_ROOT / "hard_case_rescue_audits.json").open("w") as f:
    json.dump(hard_cases, f, indent=2)

promotion_status = "offline_candidate_promoted" if rescue_correct >= 47 and rescue_regressions == 0 else "diagnostic_only_keep_notebook13_notebook22"
selected_policy = {
    "policy_name": SELECTED_POLICY_NAME,
    "status": promotion_status,
    "method": "Train/validate-calibrated L2 candidate reranker with graph/Bayes rescue continuation and conservative graph certificates over Notebook 13 traces.",
    "inputs_used": {
        "notebook13_49case": str(NOTEBOOK13_ROOT),
        "notebook22_graph_features": str(NOTEBOOK22_ROOT),
        "notebook16_graph_edges": str(GRAPH_ROOT),
        "notebook19_bayes_likelihoods": str(BAYES_ROOT),
        "partial_mlp_checkpoint": str(PARTIAL_MLP_ROOT / "best_model.pt"),
        "train_validate_synthetic_states": "Generated from DDXPlus train/validate splits only.",
    },
    "no_live_api": True,
    "no_49case_label_training_or_threshold_selection": True,
    "training_calibration": {
        "synthetic_train_states": SYNTH_TRAIN_STATES,
        "synthetic_validate_states": SYNTH_VALIDATE_STATES,
        "selected_logistic_C": best_C,
        "selected_logistic_min_score": best_threshold[0],
        "selected_logistic_min_score_margin": best_threshold[1],
    },
    "selected_policy_certificates": {
        "prior_recovery": "prior_top1 graph_rank <= 5 and Notebook13 graph_score < 0 and prior graph_score > Notebook13 graph_score",
        "graph_critic": "Notebook22 conservative graph critic: graph top differs, margin >= 1, reference graph score < 0, graph top score > 0",
        "rescue_trigger": "agent_stop and num_requests <= 3 and graph_top1_posterior < 0.80",
        "rescue_cap": MAX_RESCUE_REQUESTS,
        "rescue_accept": "post-rescue L2 reranker candidate passes validation-selected score/margin thresholds, is graph-rank <= 3, and graph support improves over reference by >0.75 or reference graph score <0; graph-rank tie-break applies within 0.02 reranker score",
    },
    "notebook13_reference": {
        "correct_count": notebook13_correct,
        "accuracy": notebook13_accuracy,
        "top3_accuracy": notebook13_top3,
        "top5_accuracy": notebook13_top5,
        "mean_requests": notebook13_mean_requests,
    },
    "notebook22_reference": {
        "correct_count": notebook22_correct,
        "accuracy": notebook22_accuracy,
        "regressions_vs_notebook13": notebook22_regressions,
    },
    "selected_policy_49case": {
        "correct_count": rescue_correct,
        "accuracy": rescue_accuracy,
        "mean_requests": rescue_mean_requests,
        "extra_requests_total": rescue_extra_requests,
        "improvements_vs_notebook13": rescue_improvements,
        "regressions_vs_notebook13": rescue_regressions,
        "changed_predictions": int((rescue_results["notebook13_predicted_pathology"] != rescue_results["rescue_predicted_pathology"]).sum()),
    },
    "paired_counts": rescue_results["win_loss_vs_notebook13"].value_counts().to_dict(),
    "promotion_rule": "Promote if selected policy reaches at least 47/49 with zero regressions against Notebook 13.",
}
with (ARTIFACT_ROOT / "selected_rescue_policy.json").open("w") as f:
    json.dump(selected_policy, f, indent=2)

resolved_run_config = {
    "notebook": "23_calibrated_graph_bayes_rescue_reranker.ipynb",
    "artifact_root": str(ARTIFACT_ROOT),
    "seed": SEED,
    "synthetic_train_nrows_loaded": SYNTH_TRAIN_NROWS,
    "synthetic_validate_nrows_loaded": SYNTH_VALIDATE_NROWS,
    "synthetic_train_states": SYNTH_TRAIN_STATES,
    "synthetic_validate_states": SYNTH_VALIDATE_STATES,
    "max_rescue_requests": MAX_RESCUE_REQUESTS,
    "total_request_cap": TOTAL_REQUEST_CAP,
    "reranker_tie_graph_eps": RERANKER_TIE_GRAPH_EPS,
    "api_usage": "none",
}
with (ARTIFACT_ROOT / "resolved_run_config.json").open("w") as f:
    json.dump(resolved_run_config, f, indent=2)

display(policy_summary)
print(json.dumps(selected_policy["selected_policy_49case"], indent=2))

## 9. Figures

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(policy_summary["system"], policy_summary["accuracy"], color=["#7a8da8", "#4f9d88", "#c97b4d"])
plt.ylim(0.80, 1.0)
plt.ylabel("Accuracy")
plt.xticks(rotation=20, ha="right")
plt.title("Notebook 23 Rescue Accuracy")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "accuracy_comparison.png", dpi=160)
plt.close()

paired_counts = rescue_results["win_loss_vs_notebook13"].value_counts().reindex(
    ["both_correct", "rescue_only_correct", "notebook13_only_correct", "both_wrong"], fill_value=0
)
plt.figure(figsize=(6, 4))
plt.bar(paired_counts.index, paired_counts.values, color="#5b7c99")
plt.xticks(rotation=20, ha="right")
plt.ylabel("Cases")
plt.title("Paired Outcomes vs Notebook 13")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "paired_outcomes.png", dpi=160)
plt.close()

plt.figure(figsize=(6, 4))
plt.hist(rescue_results["extra_requests"], bins=[-0.5, 0.5, 1.5, 2.5, 3.5], color="#8f6f9f", edgecolor="white")
plt.xlabel("Extra rescue requests")
plt.ylabel("Cases")
plt.title("Rescue Request Cost")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "extra_request_distribution.png", dpi=160)
plt.close()

hard = rescue_results[~rescue_results["notebook13_correct"]].copy()
plt.figure(figsize=(7, 4))
plt.barh(hard["case_id"], hard["rescue_correct"].astype(int), color="#c97b4d")
plt.xlabel("Correct after Notebook 23")
plt.title("Notebook 13 Misses After Rescue")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "hard_case_rescue_outcomes.png", dpi=160)
plt.close()

print("Figures saved to", FIGURE_DIR)

## 10. Final Summary And Artifact Contract

In [ ]:
required_artifacts = [
    "resolved_run_config.json",
    "synthetic_state_generation_summary.csv",
    FEATURE_CSV_NAME,
    "reranker_validation_summary.csv",
    "notebook13_rescue_case_results.csv",
    "candidate_level_49case_scores.csv",
    "rescue_trace.jsonl",
    "paired_notebook13_vs_rescue_reranker.csv",
    "hard_case_rescue_audits.json",
    "selected_rescue_policy.json",
    "rescue_policy_summary.csv",
    "figures/accuracy_comparison.png",
    "figures/paired_outcomes.png",
    "figures/extra_request_distribution.png",
    "figures/hard_case_rescue_outcomes.png",
]
missing = [name for name in required_artifacts if not (ARTIFACT_ROOT / name).exists()]
assert not missing, missing

assert notebook13_correct == 43
assert abs(notebook13_top3 - 45 / 49) < 1e-12
assert abs(notebook13_top5 - 46 / 49) < 1e-12
assert abs(notebook13_mean_requests - 6.591836734693878) < 1e-12
assert notebook22_correct == 44
assert notebook22_regressions == 0

summary_payload = {
    "notebook13_correct": notebook13_correct,
    "notebook22_correct": notebook22_correct,
    "notebook23_correct": rescue_correct,
    "notebook23_mean_requests": rescue_mean_requests,
    "extra_requests_total": rescue_extra_requests,
    "improvements_vs_notebook13": rescue_improvements,
    "regressions_vs_notebook13": rescue_regressions,
    "promotion_status": promotion_status,
}
print(json.dumps(summary_payload, indent=2))
print("Artifact contract OK:", len(required_artifacts), "files")